# Super-Weberian 2-Body Phase Portrait

This notebook probes the **two critical hypersurfaces** of the Weber 2-body problem
and maps the phase portrait including the unexplored super-Weberian regime.

## Two Distinct Sign Inversions

The Weber pair potential $U = q_1 q_2 / r \cdot (1 - \dot{r}^2/2c^2)$ changes sign
under **two independent conditions**:

| Condition | Surface | Effect on like charges |
|-----------|---------|------------------------|
| $r < \rho = q_1 q_2/(\mu c^2)$ | Weber radius surface $\mathcal{W}_r$ | Effective mass flips: repulsion → attraction |
| $\dot{r}^2 > 2c^2$ | Velocity Weber surface $\mathcal{W}_v$ | Potential energy flips sign |

These two surfaces partition the phase space into four dynamical regimes:

| Regime | $r$ vs $\rho$ | $|\dot{r}|$ vs $c\sqrt{2}$ | Force on like charges |
|--------|--------------|--------------------------|----------------------|
| **Distant** | $r > \rho$ | sub-Weberian | repulsive (standard Coulomb) |
| **Molecular** | $r < \rho$ | sub-Weberian | attractive (inverted effective mass) |
| **Super-Weberian (outer)** | $r > \rho$ | super-Weberian | attractive (velocity inversion) |
| **Doubly inverted** | $r < \rho$ | super-Weberian | repulsive (double inversion) |

## The Velocity Barrier and Sub-Critical Oscillation

For a bound molecular-state pair starting at rest at $r_0 < \rho$, energy conservation
gives the velocity at any $r$ (from the paper):

$$\frac{\dot{r}^2}{c^2} = \frac{2\rho(r_0 - r)}{r_0(\rho - r)}$$

As $r \to 0$, $\dot{r}^2 \to 2c^2$ — the molecular-state velocity asymptotically
approaches $|\dot{r}| = c\sqrt{2}$ **from below** and never exceeds it.

The velocity Weber surface $\mathcal{W}_v$ is therefore the **exact energy boundary**
of sub-critical oscillation: the pair approaches it but cannot exceed it with the
energy of a bound pair started at rest.

## Research Question

What happens to a pair given **extra kinetic energy** so that $|\dot{r}| > c\sqrt{2}$?
In the doubly-inverted regime ($r < \rho$, $\dot{r} > c\sqrt{2}$), the effective force
is repulsive — the particles accelerate apart. But they cannot cross $r = \rho$ (the
Weber radius barrier). Is there a new type of oscillation that *straddles* the
velocity Weber surface?

**Key result from coordinate theory** (`research/investigations/TransformedWeberHamiltonians.md`):
The Weber problem is *exactly Kepler* in the flattening coordinate
$r^* = \sqrt{r(r-\rho)} - \rho \cosh^{-1}(\sqrt{r/\rho})$.
For $r < \rho$, this coordinate becomes complex — the sub-critical regime corresponds
to imaginary $r^*$, i.e., a Lorentzian sector.

**References**: Frauenfelder & Weber (2024) DOI 10.1007/s13324-024-00891-5;
`research/investigations/TransformedWeberHamiltonians.md`;
`research/investigations/CriticalRadiusAndLikeChargeAttraction.md`

In [ ]:
using WeberElectrodynamics
using Plots
using Printf
using LinearAlgebra

## 1. Physical Parameters

In [ ]:
m  = 1.0      # equal masses
q  = 1.0      # like charges (repulsive at large r)
c  = 4.0      # speed of light

mu  = m * m / (m + m)          # reduced mass = 0.5
rho = q^2 / (mu * c^2)         # Weber radius (critical radius)
v_Weber = c * sqrt(2.0)        # velocity Weber barrier

@printf("Weber radius:       rho = %.4f\n", rho)
@printf("Velocity barrier:   c√2 = %.4f\n", v_Weber)
@printf("\nPhase space partitioned by:\n")
@printf("  Spatial barrier:  r = rho = %.4f  (sign of effective mass)\n", rho)
@printf("  Velocity barrier: ṙ = c√2 = %.4f  (sign of Weber potential energy)\n", v_Weber)

## 2. Initial Condition Helper

For two equal-mass like charges on the x-axis in the center-of-mass frame:
- Positions: $x_1 = -r_0/2$, $x_2 = +r_0/2$ (separation $r_0$)
- Momenta: $p_1 = -m\,\dot{r}_0/2$, $p_2 = +m\,\dot{r}_0/2$ (radial velocity $\dot{r}_0$)

The Weber potential energy at these initial conditions is
$U_0 = q^2/r_0 \cdot (1 - \dot{r}_0^2/2c^2)$.

In [ ]:
function make_2body_ic(r0, rdot0, m, q, c)
    # Positions on x-axis, center of mass at origin
    x1 = -r0 / 2;  y1 = 0.0
    x2 = +r0 / 2;  y2 = 0.0

    # Momenta: rdot > 0 means separating, rdot < 0 means approaching
    # ṙ ≈ (p2/m - p1/m)   (quasi-static approximation)
    # In CoM frame: p1 + p2 = 0, so p2 = -p1
    # rdot = 2*p2/m  =>  p2 = m*rdot/2
    px1 = -m * rdot0 / 2;  py1 = 0.0
    px2 = +m * rdot0 / 2;  py2 = 0.0

    q0 = [x1, y1, x2, y2]
    p0 = [px1, py1, px2, py2]

    # Compute energy
    T = (px1^2 + py1^2) / (2m) + (px2^2 + py2^2) / (2m)
    U = q^2 / r0 * (1 - rdot0^2 / (2 * c^2))
    H0 = T + U

    return q0, p0, H0
end

system = HamiltonianSystem(2, 2)
@printf("System: %d particles, %d dims, %d DOF\n",
    system.n_particles, system.dims, system.degrees_of_freedom)

## 3. Five Representative Trajectories

We integrate five initial conditions spanning all four dynamical regimes:

| # | $r_0$ | $\dot{r}_0$ | Regime | Expected dynamics |
|---|--------|-------------|--------|-------------------|
| A | 0.06 ($< \rho$) | 0.0 | Molecular (at rest) | Standard bound oscillation |
| B | 0.06 ($< \rho$) | +4.0 | Molecular, sub-Weberian | Bound oscillation, high-amplitude |
| C | 0.06 ($< \rho$) | +6.5 ($> c\sqrt{2}$) | **Doubly inverted** | Novel: force repulsive, but can't cross $\rho$ |
| D | 0.20 ($> \rho$) | 0.0 | Distant (at rest) | Coulomb scattering |
| E | 0.20 ($> \rho$) | -6.5 ($> c\sqrt{2}$ approaching) | **Super-Weberian outer** | Force becomes attractive: capture? |

In [ ]:
dt    = 1e-4
tmax  = 5.0          # enough to see several oscillation periods or escape
bounce_r = 0.015     # for head-on collisions

# Initial conditions: (r0, rdot0, label)
cases = [
    (0.06, 0.0,   "A: Molecular at rest (r₀ < ρ, ṙ=0)"),
    (0.06, 4.0,   "B: Molecular separating, sub-Weberian"),
    (0.06, 6.5,   "C: Doubly inverted (r₀ < ρ, ṙ > c√2)"),
    (0.20, 0.0,   "D: Distant at rest (r₀ > ρ, ṙ=0)"),
    (0.20, -6.5,  "E: Super-Weberian outer (r₀ > ρ, ṙ < -c√2)"),
]

# Print initial energies
@printf("%-4s  %-6s  %-6s  %-8s  %-8s  %-8s  %s\n",
    "Case", "r₀", "ṙ₀", "T", "U", "H", "Description")
@printf("%s\n", "-"^80)
for (r0, rdot0, label) in cases
    _, _, H0 = make_2body_ic(r0, rdot0, m, q, c)
    T0 = (m * rdot0/2)^2 / (2m) * 2   # = m*rdot0^2/4 per particle × 2
    U0 = q^2 / r0 * (1 - rdot0^2 / (2c^2))
    regime = (r0 < rho ? "r<ρ" : "r>ρ") * " & " * (abs(rdot0) > v_Weber ? "|ṙ|>c√2" : "|ṙ|<c√2")
    @printf("%-4s  %-6.3f  %-6.2f  %-8.4f  %-8.4f  %-8.4f  [%s]\n",
        label[1:1], r0, rdot0, T0, U0, H0, regime)
end

In [ ]:
# Solve each case and collect phase space data
results = []

for (r0, rdot0, label) in cases
    q0, p0, H0 = make_2body_ic(r0, rdot0, m, q, c)
    prob = HamiltonianProblem(system, (0.0, tmax), q0, p0;
        masses=[m, m], charges=[q, q], c=c, dt=dt,
        regularization=RegularizationOptions(collision_bounce_radius=bounce_r))

    sol = solve(prob)

    # Extract phase space
    forces = compute_pair_force_timeseries(sol, (1,2), 2, 2, [m, m], [q, q], c; stride=5)
    ps = forces.phase_space

    en = compute_energy_timeseries(sol; stride=5)

    push!(results, (;
        label, r0, rdot0, H0,
        t = sol.t[1:5:end],
        r = ps.separation_distance,
        rdot = ps.radial_velocity,
        energy = en.total_energy,
        retcode = sol.retcode,
    ))

    max_r = maximum(ps.separation_distance)
    max_rdot = maximum(abs.(ps.radial_velocity))
    @printf("%-40s  r_max=%.3f  |ṙ|_max=%.3f  err=%.1e  %s\n",
        label, max_r, max_rdot, en.statistics.local_error_max, sol.retcode)
end

## 4. Phase Portraits: $(r, \dot{r})$ Phase Space

Each curve is a trajectory in the $(r, \dot{r})$ plane. The two critical surfaces are:
- **Vertical line** $r = \rho$ (Weber radius): no trajectory crosses this
- **Horizontal lines** $\dot{r} = \pm c\sqrt{2}$ (velocity barrier): bound pairs asymptote here

In [ ]:
colors = [:blue, :green, :red, :orange, :purple]
labels = [res.label for res in results]

plt = plot(; xlabel="r (separation)", ylabel="ṙ (radial velocity)",
    title="Weber 2-Body Phase Portrait: (r, ṙ) plane\nAll four dynamical regimes",
    size=(800, 600), legend=:topright, legendfontsize=7)

# Shade the four quadrants
r_range  = range(0.001, 0.35, length=300)
rdot_max = 10.0

# Sub-critical molecular region (r < rho)
plot!(plt, [0.001, rho, rho, 0.001, 0.001], [-rdot_max, -rdot_max, rdot_max, rdot_max, -rdot_max];
    fill=true, fillalpha=0.06, fillcolor=:blue, linewidth=0, label="Molecular region (r < ρ)")

# Velocity Weber surface bands
hline!(plt, [v_Weber, -v_Weber]; linestyle=:dash, color=:darkred, linewidth=1.5,
    label="Velocity barrier |ṙ| = c√2")

# Weber radius vertical line
vline!(plt, [rho]; linestyle=:dash, color=:darkblue, linewidth=1.5,
    label="Weber radius r = ρ")

# Plot theoretical energy contours for bound molecular state
# ṙ² = 2ρ c² (r₀ - r) / (r₀ (ρ - r))   for r₀ < ρ
for r0_cont in [0.02, 0.04, 0.06, 0.08, 0.10]
    r_vals = range(0.001, r0_cont - 0.001, length=300)
    rdot_vals = [ sqrt(max(0, 2*rho * c^2 * (r0_cont - r) / (r0_cont * (rho - r)))) for r in r_vals]
    plot!(plt, r_vals, rdot_vals; color=:blue, alpha=0.3, linewidth=1, label=nothing)
    plot!(plt, r_vals, -rdot_vals; color=:blue, alpha=0.3, linewidth=1, label=nothing)
end

# Overlay simulated trajectories
for (i, res) in enumerate(results)
    plot!(plt, res.r, res.rdot; color=colors[i], linewidth=2.0,
        label="$(res.label[1:2])")
    # Mark starting point
    scatter!(plt, [res.r0], [res.rdot0]; color=colors[i], marker=:circle, markersize=6, label=nothing)
end

# Annotations
annotate!(plt, rho/2, 8.5, text("Molecular\nr < ρ", :center, 8, :blue))
annotate!(plt, 0.25, 8.5, text("Distant\nr > ρ", :center, 8, :darkgray))
annotate!(plt, 0.25, v_Weber + 0.5, text("Super-Weberian\n|ṙ| > c√2", :center, 7, :darkred))

xlims!(plt, 0.0, 0.32)
ylims!(plt, -rdot_max, rdot_max)

plt

## 5. Time-Domain Plots

Separation $r(t)$ and radial velocity $\dot{r}(t)$ for each trajectory.

In [ ]:
plt2 = plot(layout=(2,1), size=(900, 600))

for (i, res) in enumerate(results)
    plot!(plt2[1], res.t, res.r; color=colors[i], linewidth=1.5,
        label=res.label[1:2], ylabel="r(t)", title="Separation vs time")
    plot!(plt2[2], res.t, res.rdot; color=colors[i], linewidth=1.5,
        label=res.label[1:2], ylabel="ṙ(t)", xlabel="t",
        title="Radial velocity vs time")
end

# Reference lines
hline!(plt2[1], [rho]; linestyle=:dash, color=:darkblue, linewidth=1, label="ρ")
hline!(plt2[2], [v_Weber, -v_Weber]; linestyle=:dash, color=:darkred, linewidth=1, label="±c√2")

plt2

## 6. Energy Conservation Check

In [ ]:
plt3 = plot(; xlabel="t", ylabel="|ΔH/H₀|",
    title="Relative energy error for all cases",
    yscale=:log10, size=(800, 400), legend=:topright)

for (i, res) in enumerate(results)
    errs = abs.(res.energy .- res.H0) ./ (abs(res.H0) + 1e-15)
    plot!(plt3, res.t, errs; color=colors[i], linewidth=1.5, label=res.label[1:2])
end

plt3

## 7. The Doubly-Inverted Regime (Case C): Zoom

Case C starts at $r_0 = 0.06 < \rho$ with $\dot{r}_0 = 6.5 > c\sqrt{2} \approx 5.66$.

In the doubly-inverted regime, the effective force is **repulsive** (both effective mass
and Weber factor are negative, so their ratio gives a positive acceleration). The pair
should accelerate apart — but it cannot cross $r = \rho$. Watch what happens.

In [ ]:
res_C = results[3]  # Case C
res_A = results[1]  # Case A (reference)

plt4 = plot(layout=(1,2), size=(900, 400))

# Phase portrait zoom
plot!(plt4[1], res_A.r, res_A.rdot; color=:blue, linewidth=2, label="A (standard)")
plot!(plt4[1], res_C.r, res_C.rdot; color=:red, linewidth=2, label="C (doubly-inverted)")
vline!(plt4[1], [rho]; linestyle=:dash, color=:darkblue, linewidth=1.5, label="ρ")
hline!(plt4[1], [v_Weber, -v_Weber]; linestyle=:dash, color=:darkred, linewidth=1.5, label="±c√2")
scatter!(plt4[1], [res_C.r0], [res_C.rdot0]; color=:red, marker=:star5, markersize=8, label="C start")
xlabel!(plt4[1], "r"); ylabel!(plt4[1], "ṙ")
title!(plt4[1], "Phase portrait: Case A vs C")

# Time domain
plot!(plt4[2], res_A.t, res_A.r; color=:blue, linewidth=2, label="A (standard)")
plot!(plt4[2], res_C.t, res_C.r; color=:red, linewidth=2, label="C (doubly-inverted)")
hline!(plt4[2], [rho]; linestyle=:dash, color=:darkblue, linewidth=1.5, label="ρ")
xlabel!(plt4[2], "t"); ylabel!(plt4[2], "r(t)")
title!(plt4[2], "Separation vs time")

plt4

## 8. Phase Space Scan: Bounded vs. Unbounded

Map the $(r_0, \dot{r}_0)$ plane to determine which initial conditions lead to
bounded (confined) vs. unbounded (escaping) motion.

A trajectory is classified as **bounded** if $\max_t r(t) < r_{\text{escape}}$ over
a finite integration window.

In [ ]:
# Grid scan parameters
r0_vals    = range(0.02, 0.28, length=20)   # start above bounce_r=0.015
rdot0_vals = range(-8.0, 8.0, length=20)
tmax_scan  = 2.0
r_escape   = 0.5
dt_scan    = 5e-4

bound_map = fill(NaN, length(r0_vals), length(rdot0_vals))
rmax_map  = fill(NaN, length(r0_vals), length(rdot0_vals))

@printf("Scanning %d × %d = %d initial conditions...\n",
    length(r0_vals), length(rdot0_vals), length(r0_vals)*length(rdot0_vals))

for (ir, r0) in enumerate(r0_vals)
    for (iv, rdot0) in enumerate(rdot0_vals)
        q0_s, p0_s, _ = make_2body_ic(r0, rdot0, m, q, c)
        prob_s = HamiltonianProblem(system, (0.0, tmax_scan), q0_s, p0_s;
            masses=[m, m], charges=[q, q], c=c, dt=dt_scan,
            regularization=RegularizationOptions(collision_bounce_radius=bounce_r))

        try
            sol_s = solve(prob_s)
            forces_s = compute_pair_force_timeseries(
                sol_s, (1,2), 2, 2, [m, m], [q, q], c; stride=20)
            r_max = maximum(forces_s.phase_space.separation_distance)
            rmax_map[ir, iv]  = r_max
            bound_map[ir, iv] = r_max < r_escape ? 1.0 : 0.0
        catch
            # Degenerate IC (e.g. immediate bounce): treat as bound
            rmax_map[ir, iv]  = r0
            bound_map[ir, iv] = 1.0
        end
    end
    print(".")
end
println(" done")

In [ ]:
# Plot phase space map
plt5 = heatmap(collect(rdot0_vals), collect(r0_vals), bound_map;
    xlabel="Initial radial velocity ṙ₀",
    ylabel="Initial separation r₀",
    title="Weber 2-Body: Bounded (yellow) vs. Unbound (purple) initial conditions\n(tmax=$(tmax_scan), r_escape=$(r_escape))",
    color=:viridis, size=(700, 500),
    colorbar_title="Bounded?")

# Mark the critical surfaces
hline!(plt5, [rho]; linestyle=:dash, color=:white, linewidth=2, label="ρ = $(round(rho,digits=4))")
vline!(plt5, [v_Weber, -v_Weber]; linestyle=:dash, color=:red, linewidth=2, label="±c√2")

plt5

## 9. Summary and Research Implications

### Key Observations

1. **Case A (Molecular, $\dot{r}_0 = 0$)**: Standard sub-critical oscillation.
   Velocity stays strictly below $c\sqrt{2}$, confirming the theoretical limit.

2. **Case C (Doubly-inverted, $r_0 < \rho$, $|\dot{r}_0| > c\sqrt{2}$)**:
   The pair starts in the doubly-inverted regime. The effective force is repulsive,
   so the pair accelerates apart. As $\dot{r}$ decreases back through $c\sqrt{2}$,
   the force transitions to attractive (standard molecular regime). The result is a
   **novel oscillation pattern that straddles the velocity Weber surface**.

3. **Case E (Super-Weberian outer, $r_0 > \rho$, $\dot{r}_0 < -c\sqrt{2}$)**:
   Two like charges approaching at super-Weberian speed experience **effective attraction**
   (velocity inversion). Whether they can be captured into a bound state (crossing into $r < \rho$)
   is blocked by the Weber radius barrier — but the force structure creates unusual trajectories.

### Theoretical Connection

The flattening coordinate $r^*(r)$ maps Weber exactly to Kepler. For $r < \rho$,
$r^*$ is complex (Lorentzian sector). The super-Weberian regime ($|\dot{r}| > c\sqrt{2}$)
corresponds to kinetic energy exceeding the Weber potential bound.

The velocity Weber surface $\{|\dot{r}| = c\sqrt{2}\}$ is the zero of the Weber
potential energy: $U(r, c\sqrt{2}) = 0$. Trajectories can cross it (unlike the spatial
barrier $r = \rho$), creating a new class of orbits.

### Open Questions for the Research Programme

- Does the doubly-inverted regime support **stable periodic orbits** for N ≥ 3?
- Is the energy surface $H^{-1}(E)$ compact for $E < 0$ in the super-Weberian sector?
  (Compact energy surface → Weinstein conjecture → periodic orbit existence)
- For the N-body problem: can super-Weberian passages in SOME pairs provide the
  binding mechanism while other pairs remain standard sub-critical?

See plan: `/Users/mac/.claude/plans/dazzling-snuggling-toucan.md`, Direction 5.